<a href="https://colab.research.google.com/github/GilangAngga12/Pemrosesan-teks-praktik/blob/main/Pemrosesan_teks29okt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# 1. Instalasi & Update Package
%pip install -q --upgrade yt-dlp pandas

import os
import time
from datetime import datetime
import pandas as pd
from yt_dlp import YoutubeDL

# --- KONFIGURASI UTAMA ---
OUTPUT_FILE = "crowling scaling YouTube pemrosesan teks.csv"
COOKIES_FILE = "cookies.txt"      # File cookie (jika diunggah di folder kerja)

TARGET_TOTAL_VIDEOS = 2000        # Target jumlah video yang di-crawl
MAX_VIDEOS_PER_SOURCE = 400       # Batas max video per channel
MAX_COMMENTS_PER_VIDEO = 80       # Batas max komentar per video
REQUEST_DELAY = 1.0               # Jeda request (detik) - dinaikkan agar lebih aman dari rate limit
CATEGORY = "politik"

# --- KATA KUNCI POLITIK (UNTUK FILTERING TEXT) ---
KEYWORDS = {
    "politik": [
        # Lembaga & Jabatan
        "politik", "pemilu", "pilkada", "capres", "cawapres", "presiden", "wapres",
        "dpr", "mpr", "dpd", "mk", "kpu", "bawaslu", "kpk", "pemerintah", "kabinet",
        "menteri", "istana", "parlemen", "senayan", "mahkamah", "konstitusi",

        # Partai Politik & Koalisi
        "partai", "pdip", "gerindra", "golkar", "nasdem", "pks", "demokrat",
        "pkb", "psi", "pan", "ppp", "perindo", "hanura", "gelora", "koalisi", "oposisi",

        # Isu & Kebijakan
        "kebijakan", "uu", "ruu", "hukum", "korupsi", "skandal", "dinasti",
        "demokrasi", "hak asasi", "kampanye", "orasi", "debat", "suara", "voter",
        "anggaran", "apbn", "subsidi", "blt", "pajak", "sengketa", "putusan",

        # Reaksi & Sentimen Opini Publik
        "setuju", "tolak", "dukung", "pilih", "revolusi", "rakyat", "bohong", "hoaks"
    ]
}

# --- SUMBER REFERENSI YOUTUBE (15 CHANNEL BERITA & POLITIK) ---
SOURCES = {
    "kompas_tv": {"politik": "https://www.youtube.com/@KompasTV/videos"},
    "cnn_indonesia": {"politik": "https://www.youtube.com/@CNNindonesia/videos"},
    "tvone": {"politik": "https://www.youtube.com/@tvOneNews/videos"},
    "mata_najwa": {"politik": "https://www.youtube.com/@MataNajwa/videos"},
    "official_inews": {"politik": "https://www.youtube.com/@officialinews/videos"},
    "metro_tv": {"politik": "https://www.youtube.com/@METROTV/videos"},
    "berita_satu": {"politik": "https://www.youtube.com/@OfficialBTV/videos"},
    "cnbc_indonesia": {"politik": "https://www.youtube.com/@CNBCIndonesia/videos"},
    "liputan6": {"politik": "https://www.youtube.com/@Liputan6SCTV/videos"},
    "tempo": {"politik": "https://www.youtube.com/@TempoCo/videos"},
    "narasi": {"politik": "https://www.youtube.com/@narasi.tv/videos"},
    "tribunnews": {"politik": "https://www.youtube.com/@tribunnews/videos"},
    "rocky_gerung": {"politik": "https://www.youtube.com/@RockyGerungOfficial_/videos"},
    "refly_harun": {"politik": "https://www.youtube.com/@ReflyHarunChannel/videos"},
    "dpr_ri": {"politik": "https://www.youtube.com/@DPRRIOfficial/videos"}
}

# --- HEADER KOLOM TUNGGAL UNTUK PEMROSESAN TEKS ---
COLUMNS = [
    "source", "category", "channel_name", "video_title", "video_url",
    "video_date", "view_count", "video_like_count", "comment_count",
    "duration_seconds", "video_keywords", "video_description",
    "comment_author", "comment_text", "comment_likes", "scraped_at"
]

existing_urls = set()
if os.path.exists(OUTPUT_FILE):
    try:
        df_exist = pd.read_csv(OUTPUT_FILE)
        existing_urls = set(df_exist["video_url"].dropna().unique())
        print(f"📁 File '{OUTPUT_FILE}' ditemukan. Ada {len(existing_urls)} video yang sudah terdata sebelumnya.")
    except Exception:
        pd.DataFrame(columns=COLUMNS).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
else:
    pd.DataFrame(columns=COLUMNS).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# --- HELPER FUNCTIONS ---
def match_keywords(text, category):
    if not text:
        return []
    text_lower = str(text).lower()
    return [kw for kw in KEYWORDS.get(category, []) if kw.lower() in text_lower]

def get_base_ydl_opts():
    """Membuat konfigurasi dasar yt-dlp yang dilengkapi mekanisme penanganan anti-bot."""
    opts = {
        'skip_download': True,
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': True,
        # Mengubah identitas client ke mobile android/ios untuk menghindari blokir bot
        'extractor_args': {
            'youtube': {
                'player_client': ['android', 'ios']
            }
        }
    }
    # Jika file cookies.txt tersedia, sertakan secara otomatis
    if os.path.exists(COOKIES_FILE):
        opts['cookiefile'] = COOKIES_FILE
    return opts

def get_video_links(channel_url, limit=300):
    ydl_opts = get_base_ydl_opts()
    ydl_opts.update({
        'extract_flat': 'in_playlist',
        'playlistend': limit,
    })

    links = []
    with YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(channel_url, download=False)
            if info and 'entries' in info:
                for entry in info['entries']:
                    if entry:
                        v_id = entry.get('id')
                        v_url = entry.get('url')
                        if v_url and v_url.startswith('http'):
                            links.append(v_url)
                        elif v_id:
                            links.append(f"https://www.youtube.com/watch?v={v_id}")
        except Exception as e:
            print(f"     ⚠️ ERROR mengambil link dari {channel_url}: {e}")
    return links

def scrape_video_data(video_url, source, category):
    ydl_opts = get_base_ydl_opts()
    ydl_opts.update({
        'getcomments': True,
        'max_comments': MAX_COMMENTS_PER_VIDEO,
    })

    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)
            if not info:
                return []

            title = info.get('title', '')
            description = info.get('description', '')

            matched_v_keywords = match_keywords(f"{title} {description}", category)
            if not matched_v_keywords:
                return []

            raw_date = info.get('upload_date', '')
            date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}" if len(raw_date) == 8 else raw_date

            common_info = {
                "source": source,
                "category": category,
                "channel_name": info.get('uploader', ''),
                "video_title": title,
                "video_url": video_url,
                "video_date": date,
                "view_count": info.get('view_count', 0),
                "video_like_count": info.get('like_count', 0),
                "comment_count": info.get('comment_count', 0),
                "duration_seconds": info.get('duration', 0),
                "video_keywords": ", ".join(matched_v_keywords),
                "video_description": str(description)[:300].replace('\n', ' '),
                "scraped_at": datetime.now().isoformat(),
            }

            rows = []
            raw_comments = info.get('comments', []) or []

            if raw_comments:
                for c in raw_comments:
                    row = common_info.copy()
                    row.update({
                        "comment_author": c.get('author', ''),
                        "comment_text": str(c.get('text', '')).replace('\n', ' '),
                        "comment_likes": c.get('like_count', 0),
                    })
                    rows.append(row)
            else:
                row = common_info.copy()
                row.update({
                    "comment_author": "",
                    "comment_text": "",
                    "comment_likes": 0,
                })
                rows.append(row)

            return rows

    except Exception as e:
        print(f"     ⚠️ ERROR request ({video_url}): {e}")
        return []

def save_to_csv(rows):
    df_chunk = pd.DataFrame(rows)
    if not df_chunk.empty:
        df_chunk.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8-sig")

# --- EKSEKUSI CRAWLING UTAMA ---
total_saved_videos = len(existing_urls)
print(f"🚀 Memulai crawling & scaling ke 1 file CSV ('{OUTPUT_FILE}')...")

if os.path.exists(COOKIES_FILE):
    print(f"🍪 File '{COOKIES_FILE}' terdeteksi dan digunakan untuk autentikasi.")
else:
    print(f"ℹ️ File '{COOKIES_FILE}' tidak ditemukan. Menggunakan spoofing client Android/iOS.")

for source_name, categories in SOURCES.items():
    if total_saved_videos >= TARGET_TOTAL_VIDEOS:
        print(f"\n🎯 Target {TARGET_TOTAL_VIDEOS} video berhasil dicapai!")
        break

    for cat_name, cat_url in categories.items():
        if total_saved_videos >= TARGET_TOTAL_VIDEOS:
            break

        print("\n" + "=" * 70)
        print(f"CHANNEL : {source_name} | TOTAL VIDEO: {total_saved_videos}/{TARGET_TOTAL_VIDEOS}")
        print(f"URL     : {cat_url}")
        print("=" * 70)

        links = get_video_links(cat_url, limit=MAX_VIDEOS_PER_SOURCE)
        print(f"Mengecek {len(links)} link video terbaru...")

        for index, url in enumerate(links, start=1):
            if total_saved_videos >= TARGET_TOTAL_VIDEOS:
                break

            if url in existing_urls:
                continue

            rows = scrape_video_data(video_url=url, source=source_name, category=cat_name)

            if rows:
                save_to_csv(rows)
                existing_urls.add(url)
                total_saved_videos += 1

                print(f"[{total_saved_videos}/{TARGET_TOTAL_VIDEOS}] 💾 DISIMPAN ({len(rows)} baris data): {rows[0]['video_title'][:40]}...")
            else:
                print(f"  [x] Tidak ada keyword cocok / Video dilewati")

            time.sleep(REQUEST_DELAY)

# --- RINGKASAN AKHIR ---
print("\n" + "=" * 50)
print(f"CRAWLING & SCALING SELESAI!")
print(f"• File Output CSV : {OUTPUT_FILE}")
print(f"• Total Video     : {total_saved_videos} video")
print("=" * 50)

df_result = pd.read_csv(OUTPUT_FILE)
df_result.head()

📁 File 'crowling scaling YouTube pemrosesan teks.csv' ditemukan. Ada 6 video yang sudah terdata sebelumnya.
🚀 Memulai crawling & scaling ke 1 file CSV ('crowling scaling YouTube pemrosesan teks.csv')...
ℹ️ File 'cookies.txt' tidak ditemukan. Menggunakan spoofing client Android/iOS.

CHANNEL : kompas_tv | TOTAL VIDEO: 6/2000
URL     : https://www.youtube.com/@KompasTV/videos
Mengecek 400 link video terbaru...
[7/2000] 💾 DISIMPAN (1 baris data): Listrik PN Jaktim Padam Saat Sidang Roy ...
[8/2000] 💾 DISIMPAN (3 baris data): Prabowo Resmikan 204 Satuan Baru di HUT ...
[9/2000] 💾 DISIMPAN (1 baris data): [FULL] Presiden Prabowo Subianto Pimpin ...
[10/2000] 💾 DISIMPAN (8 baris data): FULL! Kronologi Jelang Putusan MK soal I...
[11/2000] 💾 DISIMPAN (25 baris data): FULL! Prediksi Gibranisti-Pakar Jelang P...
[12/2000] 💾 DISIMPAN (1 baris data): Gagah! Prabowo Naik Rantis Maung Cek Pas...
[13/2000] 💾 DISIMPAN (3 baris data): Potret SBY Pakai Baret Hijau di HUT ke-8...
[14/2000] 💾 DISIMPAN (1

ERROR: [youtube:tab] @CNNindonesia/videos: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


Mengecek 0 link video terbaru...

CHANNEL : tvone | TOTAL VIDEO: 363/2000
URL     : https://www.youtube.com/@tvOneNews/videos
Mengecek 400 link video terbaru...
[364/2000] 💾 DISIMPAN (1 baris data): Momen Presiden Prabowo Pimpin Upacara Pe...
[365/2000] 💾 DISIMPAN (1 baris data): Indonesia Siap Tempur Hadapi Thailand, S...
[366/2000] 💾 DISIMPAN (6 baris data): Jaksa Tanggapi Eksepsi Roy Suryo di PN J...
[367/2000] 💾 DISIMPAN (8 baris data): MK Akan Bacakan Putusan Uji Keabsahan Sy...
  [x] Tidak ada keyword cocok / Video dilewati
  [x] Tidak ada keyword cocok / Video dilewati
  [x] Tidak ada keyword cocok / Video dilewati
  [x] Tidak ada keyword cocok / Video dilewati
  [x] Tidak ada keyword cocok / Video dilewati
[368/2000] 💾 DISIMPAN (1 baris data): Gubernur DKI Pramono Tinjau Pengelolaan ...
[369/2000] 💾 DISIMPAN (1 baris data): Tanpa Parade Defile Alutsista, Peringata...
[370/2000] 💾 DISIMPAN (3 baris data): Presiden Prabowo Ingatkan TNI Harus Teru...
[371/2000] 💾 DISIMPAN (19 bari

ERROR: [youtube:tab] @CNBCIndonesia/videos: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


Mengecek 0 link video terbaru...

CHANNEL : liputan6 | TOTAL VIDEO: 1008/2000
URL     : https://www.youtube.com/@Liputan6SCTV/videos


...

ERROR: [youtube:tab] @RockyGerungOfficial_/videos: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


Mengecek 0 link video terbaru...

CHANNEL : refly_harun | TOTAL VIDEO: 1378/2000
URL     : https://www.youtube.com/@ReflyHarunChannel/videos


ERROR: [youtube:tab] @ReflyHarunChannel/videos: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


Mengecek 0 link video terbaru...

CHANNEL : dpr_ri | TOTAL VIDEO: 1378/2000
URL     : https://www.youtube.com/@DPRRIOfficial/videos
Mengecek 400 link video terbaru...
[1379/2000] 💾 DISIMPAN (71 baris data): DPR SEPEKAN: 2-7 JUNI 2026...
[1380/2000] 💾 DISIMPAN (157 baris data): REWIND DPR REPUBLIK INDONESIA 2025...
[1381/2000] 💾 DISIMPAN (144 baris data): EFEKTIVITAS ANGGARAN DAN PENINGKATAN SDM...
[1382/2000] 💾 DISIMPAN (26 baris data): PERIZINAN RUMIT, NELAYAN TERJEPIT DPR DO...
[1383/2000] 💾 DISIMPAN (24 baris data): PENDAMPINGAN HUKUM PMI DI LUAR NEGERI: D...
[1384/2000] 💾 DISIMPAN (36 baris data): OPTIMALISASI TEKNOLOGI DALAM MITIGASI BE...
[1385/2000] 💾 DISIMPAN (13 baris data): PHK MASSAL MASIH MENGHANTUI, DPR CARI JA...
[1386/2000] 💾 DISIMPAN (5 baris data): PEMERATAAN MASA TUNGGU HAJI, DPR INGATKA...
[1387/2000] 💾 DISIMPAN (5 baris data): SOLIDARITAS INDONESIA UNTUK PALESTINA: K...
[1388/2000] 💾 DISIMPAN (2 baris data): KOMISI V DPR RI DORONG PEMBAHASAN RUU DA...
[1389/2000] 💾 

,source,category,channel_name,video_title,video_url,video_date,view_count,video_like_count,comment_count,duration_seconds,video_keywords,video_description,comment_author,comment_text,comment_likes,scraped_at
0,tvone,politik,tvOneNews,"VIRAL! Gagal Mencapai Target, 4 ABK Dianiaya D...",https://www.youtube.com/watch?v=t3Qo9sbh6ow,2026-10-05,45,2.0,0,272,pan,"Kabar Siang, https://www.tvOnenews.com - VIRAL...",2026-10-05T04:53:42.028779,NaN,NaN,0
1,tvone,politik,tvOneNews,"18 Pelaku Penganiayaan ABK Digiring Polisi, Ko...",https://www.youtube.com/watch?v=lRq38xALOLI,2026-10-05,276,5.0,0,262,menteri,"Kabar Siang, https://www.tvOnenews.com - 18 Pe...",2026-10-05T04:53:44.601239,NaN,NaN,0
2,tvone,politik,tvOneNews,"Jasad Pria Tewas di Depan Minimarket, Polisi M...",https://www.youtube.com/watch?v=zmWckEPxmw0,2026-10-05,291,5.0,0,363,pan,"Kabar Siang, https://www.tvOnenews.com - Jasad...",2026-10-05T04:53:49.281784,NaN,NaN,0
3,tvone,politik,tvOneNews,KKP Cabut Izin KM Maulana 23 Usai Dugaan Penyi...,https://www.youtube.com/watch?v=P6eMupbgrhg,2026-10-05,291,3.0,0,491,menteri,"OneNews Update, https://www.tvOnenews.com - KK...",2026-10-05T04:53:51.580854,NaN,NaN,0
4,tvone,politik,tvOneNews,Warning! 22 Aplikasi Ini Berisiko Tinggi untuk...,https://www.youtube.com/watch?v=Dd2tweos0fk,2026-10-05,164,1.0,1,514,"menteri, pan","Metropolitan, https://www.tvOnenews.com - Warn...",2026-10-05T04:53:53.947017,@bryamv-m4r,😂😂😂😂😅 sudah cukup depan 2026-3000,0


In [3]:
from google.colab import files
files.download("crowling scaling YouTube pemrosesan teks.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>